# Customer Value and LTV

Temporal target construction, customer value features, regression and evaluation.

In [ ]:
# Customer value and LTV modelling
# Target: revenue in the 90 days after a historical cutoff.
def p2():
 i=2; d=ROOT/PROJECTS[i]; o=outdir(i)
 orders=read(d/'raw_data','orders.csv'); pay=read(d/'raw_data','payments.csv'); ret=read(d/'raw_data','returns.csv')
 orders['order_date']=pd.to_datetime(orders.order_date); pay['payment_date']=pd.to_datetime(pay.payment_date)
 orders=orders[orders.order_status.str.lower().isin(['completed','delivered','fulfilled'])]
 cutoff=orders.order_date.quantile(.72); horizon=cutoff+pd.Timedelta(days=90)
 hist=orders[orders.order_date<=cutoff]; fut=orders[(orders.order_date>cutoff)&(orders.order_date<=horizon)]
 g=hist.groupby('customer_id').agg(order_count=('order_id','nunique'),hist_revenue=('gross_order_value','sum'),avg_order_value=('gross_order_value','mean'),last_order=('order_date','max')).reset_index()
 g['recency_days']=(cutoff-g.last_order).dt.days; g['orders_per_30d']=g.order_count/(g.recency_days.clip(lower=30)/30); y=fut.groupby('customer_id').gross_order_value.sum().rename('future_90d_revenue'); df=g.merge(y,left_on='customer_id',right_index=True,how='left').fillna({'future_90d_revenue':0})
 df.to_csv(o/'customer_ltv_modeling_dataset.csv',index=False)
 X=df[['order_count','hist_revenue','avg_order_value','recency_days','orders_per_30d']]; y=df.future_90d_revenue
 tr,te=train_test_split(np.arange(len(df)),test_size=.25,random_state=42)
 models={'LinearRegression':LinearRegression(),'HistGradientBoosting':HistGradientBoostingRegressor(max_iter=120,max_leaf_nodes=15,learning_rate=.05,random_state=42)}; metrics={}
 for n,m in models.items(): m.fit(X.iloc[tr],np.log1p(y.iloc[tr])); pred=np.expm1(m.predict(X.iloc[te])); metrics[n]={'MAE':mean_absolute_error(y.iloc[te],pred),'RMSE':mean_squared_error(y.iloc[te],pred)**.5,'R2':r2_score(y.iloc[te],pred)}; joblib.dump(m,o.parent/'models'/f'{n.lower()}_ltv.joblib')
 pd.DataFrame(metrics).T.to_csv(o/'ltv_model_metrics.csv'); save_metrics(i,{'cutoff':str(cutoff.date()),'horizon_days':90,'test_rows':len(te),**{f'{k}_{a}':v for k,z in metrics.items() for a,v in z.items()}})
 plt.figure(); plt.hist(y,bins=40); savefig(o/'future_ltv_distribution.png','Future 90-Day Revenue Distribution')
 (d/'notebooks'/'02_customer_ltv.py').write_text("""# Customer value and LTV modelling\n# Target: revenue in the 90 days after a historical cutoff.\n"""+textwrap.dedent(__import__('inspect').getsource(p2)))
 write_notebook(i,'Customer Value and LTV','02_customer_ltv','Temporal target construction, customer value features, regression and evaluation.')
